# 5.5 怎麼獨立控制權重衰減？


一格權重w=2，這輪任務梯度恰好0，我們還想把它稍微縮小。這份額外回拉叫權重衰減，和朝答案梯度移動是兩件事。

AdamW直接把w乘上1−ηλ，再減去Adam方向的步子。η是學習率、λ是衰減強度；取η=0.1、λ=0.2，零任務梯度下應剩2×0.98=1.96。先把回拉單獨看清楚。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

w = nn.Parameter(torch.tensor([2.0]))
optimizer = torch.optim.AdamW([w], lr=0.1, weight_decay=0.2)
w.grad = torch.zeros_like(w)
optimizer.step()
print(w.detach())
assert torch.allclose(w.detach(), torch.tensor([1.96]))

輸入參數w=2，學習率0.1，衰減0.2。我們故意給任務梯度0，讓這次只觀察衰減。`torch.zeros_like(w)` 建立同形狀的零梯度tensor；`step()` 後w變 `2×(1-0.1×0.2)=1.96`。它沒有因為任務梯度為0就維持2，最後檢查可核對縮小量0.04。

這裡的0梯度和沒有梯度還不同。在本段PyTorch的AdamW實現裡，若 `.grad=None`，該參數本次會被跳過；若有一份零梯度，優化器知道它參與了這一輪，仍可應用衰減。不要把 [1.13](https://birdhackor.github.io/tiny-perceptron-vlm/1.13.html) 清梯度的None與實際算出0梯度當同一狀態，它們會影響是否更新。

若把權重平方懲罰加進任務代價，對應梯度也會進入Adam的m、v再被縮放；AdamW讓回拉與這份歷史分開。縮小有用的大權重可能暫時提高任務代價，需用驗證資料選強度。bias或正規化倍率有時另分組、不衰減，分組規則也要明說。

練習只把weight_decay改為0，先預測零任務梯度下w維持2，再跑核對，並將預期值改為2。接着保留衰減0.2、把 `w.grad` 改為None，觀察這份實現同樣不改w。兩次實驗各回答一個問題：衰減開沒開，以及參數有沒有進入本輪更新。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
